# Temporal Mining Techniques

**Objective:** Demonstrate temporal mining using trend, rolling averages, lag features, seasonality, and anomaly detection.

**Dataset:** Synthetic monthly demand time series

This notebook is Colab-ready and saves tables, metrics, and visual outputs under
`results/`. Public datasets or compact sample datasets are used so the workflow
remains reproducible.


In [ ]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
rng = np.random.default_rng(42)


In [ ]:
dates = pd.date_range("2019-01-01", periods=72, freq="MS")
t = np.arange(len(dates))
demand = 120 + 0.9 * t + 18 * np.sin(2 * np.pi * t / 12) + rng.normal(0, 5, len(t))
demand[[18, 43, 61]] += [35, -30, 40]
ts = pd.DataFrame({"date": dates, "demand": demand})
ts["rolling_mean"] = ts["demand"].rolling(6, min_periods=1).mean()
ts["lag_1"] = ts["demand"].shift(1)
ts["lag_12"] = ts["demand"].shift(12)
ts["rolling_std"] = ts["demand"].rolling(6, min_periods=2).std()
ts["z_score"] = (ts["demand"] - ts["rolling_mean"]) / ts["rolling_std"]
ts["is_anomaly"] = ts["z_score"].abs() > 2
ts.to_csv(RESULTS_DIR / "temporal_features.csv", index=False)
display(ts.head())


In [ ]:
lag_summary = pd.DataFrame(
    {
        "lag": [1, 2, 3, 6, 12],
        "correlation": [ts["demand"].corr(ts["demand"].shift(lag)) for lag in [1, 2, 3, 6, 12]],
    }
)
lag_summary.to_csv(RESULTS_DIR / "lag_correlation.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(ts["date"], ts["demand"], label="Observed")
axes[0].plot(ts["date"], ts["rolling_mean"], label="Rolling mean")
axes[0].scatter(ts.loc[ts["is_anomaly"], "date"], ts.loc[ts["is_anomaly"], "demand"], color="red", label="Anomaly")
axes[0].set_title("Temporal Pattern with Anomalies")
axes[0].legend()
sns.barplot(data=lag_summary, x="lag", y="correlation", ax=axes[1])
axes[1].set_title("Lag Correlation")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "temporal_mining_dashboard.png", dpi=180)
plt.show()
